## Configuracion del entorno

Este notebook corria originalmente en Google Colab. Para ejecutarlo localmente en
este repositorio:

1. Crea y activa el entorno virtual del proyecto (`.venv` en la raiz del repo) e
   instala las dependencias listadas en `requirements.txt`.
2. Copia `.env.example` a `.env` (en la raiz del repo) y coloca ahi tu `HF_TOKEN`
   de Hugging Face (necesario para los datasets/modelos con acceso restringido).
3. Ejecuta las celdas en orden desde este punto.

Las carpetas de datos (`training/data/raw` para entrada, `training/data/processed`
para salida) y de modelos (`model_artifacts/`) ya existen en el repo y reemplazan
el uso de Google Drive del notebook original.

In [ ]:
import os
from pathlib import Path

# Se asume que el notebook se ejecuta con su propio directorio como cwd
# (comportamiento por defecto de Jupyter/VS Code al abrir un .ipynb).
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

RAW_DIR = PROJECT_ROOT / "training" / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "training" / "data" / "processed"
MODEL_ARTIFACTS_DIR = PROJECT_ROOT / "model_artifacts"

for carpeta in (RAW_DIR, PROCESSED_DIR, MODEL_ARTIFACTS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT:           {PROJECT_ROOT}")
print(f"RAW_DIR (entrada):      {RAW_DIR}")
print(f"PROCESSED_DIR (salida): {PROCESSED_DIR}")
print(f"MODEL_ARTIFACTS_DIR:    {MODEL_ARTIFACTS_DIR}")

### Autenticacion en Hugging Face

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv(PROJECT_ROOT / ".env")

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(hf_token)
    print("Sesion de Hugging Face iniciada.")
else:
    print("HF_TOKEN no esta definido (revisa tu archivo .env). "
          "Continuo sin iniciar sesion; solo hace falta para datasets/modelos con acceso restringido.")

## **Conseguir datasets de HuggingFace**

In [ ]:
"""
Script para descargar y preprocesar datasets publicos de discurso ofensivo
en espanol, dejandolos organizados para construir el dataset de fine-tuning
multilabel (grosero / amenaza / inapropiado).

Requisitos previos:
    pip install datasets huggingface_hub pandas

Antes de correr esto necesitas autenticarte con tu token de HuggingFace
(el mismo con el que solicitaste acceso a los datasets). En este notebook
eso se hace en la celda de login() de arriba, que lee HF_TOKEN desde el
archivo .env.

IMPORTANTE: este script NO clasifica automaticamente amenazas ni doble
sentido -- eso requiere revision manual (ver seccion "REVISION MANUAL"
al final). Lo que si hace es dejarte los datos ya separados en carpetas
candidatas para que esa revision sea rapida.
"""

import os
import pandas as pd
from datasets import load_dataset

# Carpeta de salida: datos de entrada del pipeline (raw)
OUTPUT_DIR = str(RAW_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)


def guardar_csv(df: pd.DataFrame, nombre: str):
    ruta = os.path.join(OUTPUT_DIR, nombre)
    df.to_csv(ruta, index=False, encoding="utf-8")
    print(f"  -> Guardado: {ruta} ({len(df)} filas)")


# ---------------------------------------------------------------------------
# 1. OFFENDES
# ---------------------------------------------------------------------------
print("\n=== Descargando OffendES ===")
try:
    # Usamos la rama de parquet auto-generada por HuggingFace, ya que este
    # dataset usa un script de carga y no archivos CSV sueltos con nombres
    # predecibles -- pedir el parquet evita depender de ejecutar ese script.
    offendes = load_dataset("fmplaza/offendes", revision="refs/convert/parquet")
except Exception as e:
    print(f"Fallo con refs/convert/parquet ({e}), reintentando con trust_remote_code...")
    offendes = load_dataset("fmplaza/offendes", trust_remote_code=True)

# Unimos todos los splits en un solo DataFrame para trabajar mas comodo
df_offendes = pd.concat(
    [offendes[split].to_pandas() for split in offendes.keys()],
    ignore_index=True,
)

# IMPORTANTE: al venir de la rama de parquet auto-convertida, la columna
# 'label' llega como enteros (tipo ClassLabel), no como los strings
# originales 'NO'/'NOE'/'OFP'/'OFG'. Recuperamos el mapeo int -> nombre
# desde las features del dataset y lo aplicamos antes de filtrar.
label_feature = offendes["train"].features["label"]
id2label = {i: nombre for i, nombre in enumerate(label_feature.names)}
print(f"Mapeo de etiquetas detectado: {id2label}")
df_offendes["label"] = df_offendes["label"].map(id2label)

print(f"Total de filas en OffendES: {len(df_offendes)}")
print("Distribucion de etiquetas:")
print(df_offendes["label"].value_counts())

# Separacion por categoria segun el mapeo que definimos:
# OFP / OFG -> candidatos a "grosero" (revisar si alguno es amenaza real)
# NOE       -> negativos "dificiles" (groserias sin intencion ofensiva)
# NO        -> negativos puros
candidatos_grosero = df_offendes[df_offendes["label"].isin(["OFP", "OFG"])].copy()
negativos_dificiles = df_offendes[df_offendes["label"] == "NOE"].copy()
negativos_puros = df_offendes[df_offendes["label"] == "NO"].copy()

# Renombramos la columna de texto a un nombre estandar para todo el pipeline
for df_tmp in (candidatos_grosero, negativos_dificiles, negativos_puros):
    df_tmp.rename(columns={"comment": "texto"}, inplace=True)

guardar_csv(candidatos_grosero, "offendes_candidatos_grosero.csv")
guardar_csv(negativos_dificiles, "offendes_negativos_dificiles_NOE.csv")
guardar_csv(negativos_puros, "offendes_negativos_puros.csv")


# ---------------------------------------------------------------------------
# 2. SPANISH HATE SPEECH SUPERSET
# ---------------------------------------------------------------------------
print("\n=== Descargando Spanish Hate Speech Superset ===")
superset = load_dataset("manueltonneau/spanish-hate-speech-superset")
df_superset = superset["train"].to_pandas()
print(f"Total de filas en el superset: {len(df_superset)}")
print("Distribucion de labels (1 = odio, 0 = no odio):")
print(df_superset["labels"].value_counts())
print("Distribucion por sub-dataset de origen:")
print(df_superset["dataset"].value_counts())

# Filtramos primero a los sub-datasets mas cercanos a espanol
# latinoamericano/mexicano, ya que el superset mezcla variantes de Espana
# y otros paises. Ajusta esta lista si quieres incluir mas/menos fuentes.
fuentes_relevantes = ["chileno", "homomex"]
df_superset_filtrado = df_superset[df_superset["dataset"].isin(fuentes_relevantes)].copy()
print(f"\nFilas tras filtrar por fuentes {fuentes_relevantes}: {len(df_superset_filtrado)}")

candidatos_odio = df_superset_filtrado[df_superset_filtrado["labels"] == 1].copy()
negativos_superset = df_superset_filtrado[df_superset_filtrado["labels"] == 0].copy()

candidatos_odio.rename(columns={"text": "texto"}, inplace=True)
negativos_superset.rename(columns={"text": "texto"}, inplace=True)

guardar_csv(candidatos_odio, "superset_candidatos_odio_o_amenaza.csv")
guardar_csv(negativos_superset, "superset_negativos.csv")

# Ademas del pool filtrado por fuente (chileno/homomex), guardamos tambien
# el pool COMPLETO (todas las fuentes: hateval, haternet, hascosva,
# misocorpus incluidos) -- esto es porque para AMENAZA especificamente el
# regionalismo importa menos que tener suficiente volumen para encontrar
# casos reales, que ya vimos que son escasos incluso filtrando por fuente.
candidatos_odio_completo = df_superset[df_superset["labels"] == 1].copy()
candidatos_odio_completo.rename(columns={"text": "texto"}, inplace=True)
guardar_csv(candidatos_odio_completo, "superset_candidatos_odio_TODAS_FUENTES.csv")


# ---------------------------------------------------------------------------
# 3. Muestra para revision manual (estratificada por fuente, no azar puro)
# ---------------------------------------------------------------------------
print("\n=== Generando muestras para revision manual ===")

# Tomamos una muestra estratificada por 'dataset' (chileno/homomex) de los
# candidatos "odio" del superset, para no depender de que el azar puro nos
# deje una muestra dominada por una sola fuente. Aumentamos el tamano a 220
# (en vez de 100) porque no todo "odio" es AMENAZA real -- es probable que
# una fraccion buena de estas filas termine siendo solo GROSERIA/insulto
# tras tu revision, asi que conviene partir de una muestra mas grande para
# no quedarte corto del minimo de ~30-50 ejemplos de amenaza.
TAMANO_MUESTRA = 220

if len(candidatos_odio) > 0:
    # Proporcion de cada fuente dentro del pool de candidatos
    proporciones = candidatos_odio["dataset"].value_counts(normalize=True)
    partes = []
    for fuente, proporcion in proporciones.items():
        n_fuente = max(1, round(TAMANO_MUESTRA * proporcion))
        subset_fuente = candidatos_odio[candidatos_odio["dataset"] == fuente]
        n_fuente = min(n_fuente, len(subset_fuente))  # no pedir mas de lo que hay
        partes.append(subset_fuente.sample(n=n_fuente, random_state=42))

    muestra_revision = pd.concat(partes, ignore_index=True)
    muestra_revision = muestra_revision.sample(frac=1, random_state=42).reset_index(drop=True)  # mezclar orden

    muestra_revision = muestra_revision[["texto", "dataset"]].copy()
    muestra_revision["es_grosero"] = ""       # completar a mano: 1 o 0
    muestra_revision["es_amenaza"] = ""       # completar a mano: 1 o 0
    muestra_revision["es_inapropiado"] = ""   # completar a mano: 1 o 0 (poco esperado aqui, pero dejalo disponible)

    print(f"Muestra generada: {len(muestra_revision)} filas")
    print("Distribucion por fuente en la muestra:")
    print(muestra_revision["dataset"].value_counts())

    guardar_csv(muestra_revision, "REVISAR_muestra_para_etiquetar.csv")

print("\nListo. Revisa la carpeta:", OUTPUT_DIR)
print("""
REVISION MANUAL (siguiente paso, fuera de este script):

1. Abre 'REVISAR_muestra_para_etiquetar.csv' y llena a mano las columnas
   es_amenaza / es_grosero / es_inapropiado con 1 o 0 para cada fila.
   Esta es tu fuente principal de ejemplos de AMENAZA, ya que ningun
   dataset publico los distingue automaticamente de groserias.

2. 'offendes_candidatos_grosero.csv' ya lo puedes usar casi directo como
   positivos de 'grosero' (grosero=1, amenaza=0, inapropiado=0), pero vale
   la pena revisar una muestra rapida para descartar casos raros.

3. 'offendes_negativos_dificiles_NOE.csv' son tus mejores negativos:
   groserias sin intencion ofensiva. Etiquetalos como
   grosero=0, amenaza=0, inapropiado=0. Son oro para reducir falsos
   positivos del modelo.

4. Los negativos puros (NO / superset_negativos) van directo como
   grosero=0, amenaza=0, inapropiado=0, sin revision necesaria.

5. La categoria 'inapropiado' (doble sentido) NO tiene representacion en
   estos datasets -- estos ejemplos los vas a tener que escribir a mano
   con tu equipo, como ya haviamos anticipado.

6. Al final, combina todo en un unico CSV con columnas:
   texto, grosero, amenaza, inapropiado
   listo para cargarlo con `datasets` y tokenizar para el fine-tuning.
""")

In [ ]:
"""
Script para filtrar, por palabras clave relacionadas a violencia/amenaza,
el pool de candidatos ya descargado (superset_candidatos_odio_o_amenaza.csv),
con el objetivo de aumentar la probabilidad de encontrar ejemplos reales de
AMENAZA en vez de seguir revisando muestras aleatorias con muy baja tasa
de acierto (como ya se confirmo: ~0 amenazas en 70 filas revisadas al azar).

Este script NO etiqueta nada automaticamente -- solo prioriza y reordena
las filas mas prometedoras para que tu revision manual sea mas eficiente.
La decision final de si algo es amenaza real sigue siendo tuya.

Requisitos previos:
    pip install pandas unidecode
"""

import os
import pandas as pd
from unidecode import unidecode

INPUT_DIR = str(RAW_DIR)
OUTPUT_DIR = str(RAW_DIR)

# ---------------------------------------------------------------------------
# 1. Diccionario de palabras/raices clave relacionadas a violencia explicita
# ---------------------------------------------------------------------------
# Nota: son RAICES (sin conjugar del todo) para capturar variantes por
# conjugacion verbal (matar, mato, mataria, mataria, etc.) usando "contiene".
# Ajusta/agrega terminos si tu revision manual detecta patrones que faltan.
PALABRAS_CLAVE_AMENAZA = [
    "matar", "mato", "mataste", "mataria", "mataremos", "muerete",
    "golpe", "golpear", "pegar", "pegarte", "partir la madre", "partirte",
    "destruir", "destruirte", "acabar contigo", "arruinar tu vida",
    "cuidate", "te va a pesar", "te vas a arrepentir", "vas a pagar",
    "amenaza", "amenazo", "violar", "secuestrar", "quemar tu casa",
    "te voy a", "les voy a", "le voy a", "van a ver", "vas a ver",
    "sangre", "arma", "pistola", "cuchillo", "navaja",
]


def normalizar(texto: str) -> str:
    """Normalizacion simple: minusculas + sin acentos, para que el match
    por palabra clave no falle por tildes o mayusculas."""
    if not isinstance(texto, str):
        return ""
    return unidecode(texto.lower())


def contiene_palabra_clave(texto_normalizado: str, palabras: list) -> list:
    """Devuelve la lista de palabras clave encontradas en el texto (puede
    haber mas de una), para que puedas ver por que se prioriza esa fila."""
    encontradas = [p for p in palabras if normalizar(p) in texto_normalizado]
    return encontradas


def main():
    # Preferimos el pool COMPLETO (todas las fuentes: chileno, homomex,
    # hateval, haternet, hascosva, misocorpus) porque ya confirmamos que
    # las amenazas reales son escasas incluso filtrando por region -- aqui
    # priorizamos volumen sobre regionalismo para esta categoria especifica.
    ruta_pool_completo = os.path.join(INPUT_DIR, "superset_candidatos_odio_TODAS_FUENTES.csv")
    ruta_pool_filtrado = os.path.join(INPUT_DIR, "superset_candidatos_odio_o_amenaza.csv")

    if os.path.exists(ruta_pool_completo):
        ruta_pool = ruta_pool_completo
        print("Usando el pool COMPLETO (todas las fuentes) para maximizar volumen.")
    elif os.path.exists(ruta_pool_filtrado):
        ruta_pool = ruta_pool_filtrado
        print("Aviso: no se encontro el pool completo, usando el pool filtrado "
              "(chileno/homomex). Vuelve a correr preparar_dataset.py actualizado "
              "para generar el pool completo y tener mas candidatos.")
    else:
        print(f"No se encontro ningun pool de candidatos en {INPUT_DIR}. "
              f"Corre primero el script de descarga.")
        return

    df = pd.read_csv(ruta_pool)
    print(f"Pool total de candidatos: {len(df)} filas")

    df["texto_normalizado"] = df["texto"].apply(normalizar)
    df["palabras_encontradas"] = df["texto_normalizado"].apply(
        lambda t: contiene_palabra_clave(t, PALABRAS_CLAVE_AMENAZA)
    )
    df["num_coincidencias"] = df["palabras_encontradas"].apply(len)

    # Nos quedamos solo con filas que tuvieron al menos 1 coincidencia
    df_priorizado = df[df["num_coincidencias"] > 0].copy()
    df_priorizado.sort_values("num_coincidencias", ascending=False, inplace=True)

    print(f"Filas con al menos 1 palabra clave de violencia: {len(df_priorizado)}")
    print("\nPalabras clave mas frecuentes encontradas:")
    todas_las_encontradas = [p for lista in df_priorizado["palabras_encontradas"] for p in lista]
    print(pd.Series(todas_las_encontradas).value_counts().head(15))

    # Preparamos el CSV final para revision manual, con columnas vacias
    # listas para llenar, igual que el formato que ya venias usando.
    df_salida = df_priorizado[["texto", "dataset", "palabras_encontradas", "num_coincidencias"]].copy()
    df_salida["es_grosero"] = ""
    df_salida["es_amenaza"] = ""
    df_salida["es_inapropiado"] = ""

    ruta_salida = os.path.join(OUTPUT_DIR, "REVISAR_priorizado_por_palabras_clave.csv")
    df_salida.to_csv(ruta_salida, index=False, encoding="utf-8")
    print(f"\n-> Guardado: {ruta_salida} ({len(df_salida)} filas)")

    # Tambien guardamos el resto (sin coincidencias) por si necesitas
    # negativos adicionales mas adelante -- no requieren revision.
    df_sin_coincidencia = df[df["num_coincidencias"] == 0].copy()
    df_sin_coincidencia = df_sin_coincidencia[["texto", "dataset"]].copy()
    ruta_sin = os.path.join(OUTPUT_DIR, "superset_sin_palabras_clave_violencia.csv")
    df_sin_coincidencia.to_csv(ruta_sin, index=False, encoding="utf-8")
    print(f"-> Guardado: {ruta_sin} ({len(df_sin_coincidencia)} filas, utiles como negativos adicionales)")

    print("""
SIGUIENTE PASO:
Abre 'REVISAR_priorizado_por_palabras_clave.csv'. Las filas ya vienen
ordenadas por numero de coincidencias (mas probables primero). Revisa
esta lista en vez de seguir con muestras aleatorias -- deberias encontrar
una proporcion mucho mayor de amenazas reales por fila revisada.

Recuerda: que una fila contenga una palabra clave NO significa
automaticamente que sea amenaza real (ej. "cuidate mucho" es una
despedida amistosa, no una amenaza). La columna 'palabras_encontradas'
es solo para ayudarte a priorizar, la decision sigue siendo tuya.
""")


if __name__ == "__main__":
    main()

## **Combinar datasets**

In [ ]:
"""
Script final: combina todas las fuentes de datos ya etiquetadas en un
unico CSV con columnas (texto, grosero, amenaza, inapropiado, fuente),
listo para tokenizar y usar en el fine-tuning.

Fuentes que combina:
  1. offendes_candidatos_grosero.csv   -> grosero=1 (OFP/OFG)
  2. offendes_negativos_dificiles_NOE.csv -> grosero=1 (Postura A:
     tolerancia cero, ya no se trata como negativo)
  3. offendes_negativos_puros.csv      -> negativo puro
  4. superset_negativos.csv            -> negativo puro
  5. REVISAR_priorizado_por_palabras_clave (ya etiquetado a mano)
  6. moderacion_centros_acopio.csv (dominio especifico, tanda 1)
  7. dataset_moderacion.csv (dominio especifico, tanda 2, otro LLM)
  8. my_dataset_moderacion.csv (ejemplos propios del usuario)
  9. pares_contrastivos_amenaza.csv (regla del "tono", pares contrastivos)

Requisitos previos:
    pip install pandas scikit-learn
"""

import os
import pandas as pd

INPUT_DIR = str(RAW_DIR)
OUTPUT_DIR = str(PROCESSED_DIR)

# Cuantas filas tomar como maximo de cada fuente GRANDE/generica, para no
# ahogar a los ejemplos curados y de dominio especifico. Ajusta estos
# numeros si quieres mas/menos volumen generico.
N_GROSERO_OFFENDES = 200
N_NOE_COMO_GROSERO = 200
N_NEGATIVOS_OFFENDES = 250
N_NEGATIVOS_SUPERSET = 150

RANDOM_STATE = 42


def cargar_y_muestrear(ruta, n, columnas_renombrar=None):
    if not os.path.exists(ruta):
        print(f"  [AVISO] No se encontro {ruta}, se omite esta fuente.")
        return pd.DataFrame(columns=["texto", "grosero", "amenaza", "inapropiado"])
    df = pd.read_csv(ruta)
    if columnas_renombrar:
        df = df.rename(columns=columnas_renombrar)
    if n is not None and len(df) > n:
        df = df.sample(n=n, random_state=RANDOM_STATE)
    return df


def main():
    partes = []

    # 1. Grosero de OffendES (OFP/OFG)
    print("Cargando offendes_candidatos_grosero.csv ...")
    df1 = cargar_y_muestrear(
        os.path.join(INPUT_DIR, "offendes_candidatos_grosero.csv"), N_GROSERO_OFFENDES
    )
    if len(df1) > 0:
        df1 = df1[["texto"]].copy()
        df1["grosero"], df1["amenaza"], df1["inapropiado"] = 1, 0, 0
        df1["fuente"] = "offendes_grosero"
        partes.append(df1)

    # 2. NOE reclasificado como grosero=1 (Postura A: tolerancia cero)
    print("Cargando offendes_negativos_dificiles_NOE.csv (reclasificando a grosero=1) ...")
    df2 = cargar_y_muestrear(
        os.path.join(INPUT_DIR, "offendes_negativos_dificiles_NOE.csv"), N_NOE_COMO_GROSERO
    )
    if len(df2) > 0:
        df2 = df2[["texto"]].copy()
        df2["grosero"], df2["amenaza"], df2["inapropiado"] = 1, 0, 0
        df2["fuente"] = "offendes_NOE_reclasificado"
        partes.append(df2)

    # 3. Negativos puros de OffendES
    print("Cargando offendes_negativos_puros.csv ...")
    df3 = cargar_y_muestrear(
        os.path.join(INPUT_DIR, "offendes_negativos_puros.csv"), N_NEGATIVOS_OFFENDES
    )
    if len(df3) > 0:
        df3 = df3[["texto"]].copy()
        df3["grosero"], df3["amenaza"], df3["inapropiado"] = 0, 0, 0
        df3["fuente"] = "offendes_negativo"
        partes.append(df3)

    # 4. Negativos del superset
    print("Cargando superset_negativos.csv ...")
    df4 = cargar_y_muestrear(
        os.path.join(INPUT_DIR, "superset_negativos.csv"), N_NEGATIVOS_SUPERSET
    )
    if len(df4) > 0:
        df4 = df4[["texto"]].copy()
        df4["grosero"], df4["amenaza"], df4["inapropiado"] = 0, 0, 0
        df4["fuente"] = "superset_negativo"
        partes.append(df4)

    # 5. Tu revision manual (grosero/amenaza reales, prioridad alta -- se
    # incluye COMPLETA, sin muestreo, porque es la fuente mas valiosa)
    print("Cargando REVISAR_priorizado_por_palabras_clave (etiquetado a mano) ...")
    ruta5 = os.path.join(INPUT_DIR, "REVISAR_priorizado_por_palabras_clave.csv")
    if os.path.exists(ruta5):
        df5 = pd.read_csv(ruta5)
        df5 = df5.rename(columns={
            "es_grosero": "grosero",
            "es_amenaza": "amenaza",
            "es_inapropiado": "inapropiado",
        })
        df5 = df5[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df5["fuente"] = "revision_manual_amenaza"
        partes.append(df5)
    else:
        print(f"  [AVISO] No se encontro {ruta5}, se omite esta fuente.")

    # 6. Dataset de dominio especifico (centros de acopio) -- COMPLETO,
    # es la fuente mas importante para que el modelo entienda tu contexto
    print("Cargando moderacion_centros_acopio.csv (dominio especifico) ...")
    ruta6 = os.path.join(INPUT_DIR, "moderacion_centros_acopio.csv")
    if os.path.exists(ruta6):
        df6 = pd.read_csv(ruta6)
        df6 = df6[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df6["fuente"] = "dominio_centros_acopio"
        partes.append(df6)
    else:
        print(f"  [AVISO] No se encontro {ruta6}, se omite esta fuente.")

    # 7. Dataset de dominio especifico generado por otro LLM (rondas de
    # amenaza/inapropiado/negativos con diversidad estructural) -- COMPLETO
    print("Cargando dataset_moderacion.csv (tanda adicional, otro LLM) ...")
    ruta7 = os.path.join(INPUT_DIR, "dataset_moderacion.csv")
    if os.path.exists(ruta7):
        df7 = pd.read_csv(ruta7)
        df7 = df7[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df7["fuente"] = "dominio_llm_tanda2"
        partes.append(df7)
    else:
        print(f"  [AVISO] No se encontro {ruta7}, se omite esta fuente.")

    # 8. Ejemplos redactados a mano por el usuario -- COMPLETO, alta calidad
    print("Cargando my_dataset_moderacion.csv (ejemplos propios) ...")
    ruta8 = os.path.join(INPUT_DIR, "my_dataset_moderacion.csv")
    if os.path.exists(ruta8):
        df8 = pd.read_csv(ruta8)
        df8 = df8[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df8["fuente"] = "propio_usuario"
        partes.append(df8)
    else:
        print(f"  [AVISO] No se encontro {ruta8}, se omite esta fuente.")

    # 9. Pares contrastivos de amenaza (regla del "tono") -- COMPLETO
    print("Cargando pares_contrastivos_amenaza.csv ...")
    ruta9 = os.path.join(INPUT_DIR, "pares_contrastivos_amenaza.csv")
    if os.path.exists(ruta9):
        df9 = pd.read_csv(ruta9)
        df9 = df9[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df9["fuente"] = "pares_contrastivos_tono"
        partes.append(df9)
    else:
        print(f"  [AVISO] No se encontro {ruta9}, se omite esta fuente.")

    print("Cargando negativos_casa_y_positivos_nuevos.csv ...")
    ruta10 = os.path.join(INPUT_DIR, "negativos_casa_y_positivos_nuevos.csv")
    if os.path.exists(ruta10):
        df10 = pd.read_csv(ruta10)
        df10 = df10[["texto", "grosero", "amenaza", "inapropiado"]].copy()
        df10["fuente"] = "pares_contrastivos_tono"
        partes.append(df10)
    else:
        print(f"  [AVISO] No se encontro {ruta10}, se omite esta fuente.")

    # ---------------------------------------------------------------------
    # Combinar todo
    # ---------------------------------------------------------------------
    df_final = pd.concat(partes, ignore_index=True)

    # Asegurar tipos correctos (0/1 enteros, no floats ni strings)
    for col in ["grosero", "amenaza", "inapropiado"]:
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0).astype(int)

    # Quitar duplicados exactos de texto (puede pasar entre fuentes)
    antes = len(df_final)
    df_final = df_final.drop_duplicates(subset="texto").reset_index(drop=True)
    print(f"\nFilas antes de quitar duplicados: {antes}, despues: {len(df_final)}")

    # Quitar filas con texto vacio o muy corto (ruido)
    df_final = df_final[df_final["texto"].str.strip().str.len() > 2].reset_index(drop=True)

    # Mezclar el orden (importante: no dejar el dataset agrupado por fuente)
    df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    # ---------------------------------------------------------------------
    # Reporte de balance final
    # ---------------------------------------------------------------------
    print(f"\n=== DATASET FINAL: {len(df_final)} filas ===")
    print("\nDistribucion por fuente:")
    print(df_final["fuente"].value_counts())
    print("\nPositivos por categoria:")
    print(f"  grosero=1:     {df_final['grosero'].sum()} ({df_final['grosero'].mean()*100:.1f}%)")
    print(f"  amenaza=1:     {df_final['amenaza'].sum()} ({df_final['amenaza'].mean()*100:.1f}%)")
    print(f"  inapropiado=1: {df_final['inapropiado'].sum()} ({df_final['inapropiado'].mean()*100:.1f}%)")
    print(f"\nFilas totalmente negativas (0,0,0): "
          f"{((df_final['grosero']==0)&(df_final['amenaza']==0)&(df_final['inapropiado']==0)).sum()}")

    # ---------------------------------------------------------------------
    # Guardar dataset final (sin la columna 'fuente' para el entrenamiento,
    # pero tambien una version CON fuente para que conserves trazabilidad)
    # ---------------------------------------------------------------------
    ruta_final_completa = os.path.join(OUTPUT_DIR, "dataset_final_CON_fuente.csv")
    df_final.to_csv(ruta_final_completa, index=False, encoding="utf-8")
    print(f"\n-> Guardado (con trazabilidad de fuente): {ruta_final_completa}")

    df_entrenamiento = df_final[["texto", "grosero", "amenaza", "inapropiado"]].copy()
    ruta_final = os.path.join(OUTPUT_DIR, "dataset_final_entrenamiento.csv")
    df_entrenamiento.to_csv(ruta_final, index=False, encoding="utf-8")
    print(f"-> Guardado (listo para tokenizar/entrenar): {ruta_final}")

    print("""
SIGUIENTE PASO SUGERIDO:
Este archivo ya esta listo para el split train/validation/test antes del
fine-tuning. Si el balance de categorias (arriba) se ve muy desigual
(ej. inapropiado con un porcentaje mucho menor que las otras dos),
considera ajustar los N_* al inicio del script para incluir menos
generico y asi subir el peso relativo de las categorias mas escasas,
o generar mas ejemplos de esa categoria especifica antes de continuar.
""")


if __name__ == "__main__":
    main()

## **Aplicar split estratificado**

In [ ]:
"""
Divide dataset_final_entrenamiento.csv en train/validation/test, cuidando
que las categorias minoritarias (amenaza, inapropiado) queden
representadas en los tres conjuntos -- un split aleatorio simple corre
el riesgo de dejar validation/test sin ejemplos de esas categorias, dado
lo pocos que son (61 y 33 respectivamente).

Estrategia: como sklearn no soporta estratificacion nativa para
multilabel, se construye una columna auxiliar "estrato" que resume el
caso mas "raro"/prioritario de cada fila (inapropiado > amenaza >
grosero > negativo) y se estratifica sobre esa columna. No es perfecto
para las pocas filas donde se combinan 2+ categorias a la vez, pero
garantiza cobertura minima de cada categoria en los tres splits, que es
lo que realmente importa para poder evaluar el modelo despues.

Requisitos previos:
    pip install pandas scikit-learn
"""

import os
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_DIR = str(PROCESSED_DIR)
OUTPUT_DIR = str(PROCESSED_DIR)

# Proporciones del split. 70/15/15 es un punto de partida razonable para
# un dataset de este tamano (~1,100 filas).
PROP_TRAIN = 0.70
PROP_VAL = 0.15
PROP_TEST = 0.15

RANDOM_STATE = 42


def construir_estrato(row):
    """Prioriza la categoria mas escasa para definir el 'grupo' de
    estratificacion de cada fila. Si una fila tiene varias etiquetas
    activas, se estratifica por la mas rara para asegurar que las
    categorias minoritarias no se pierdan en el split."""
    if row["inapropiado"] == 1:
        return "inapropiado"
    if row["amenaza"] == 1:
        return "amenaza"
    if row["grosero"] == 1:
        return "grosero"
    return "negativo"


def reportar_balance(nombre, df):
    total = len(df)
    print(f"\n--- {nombre}: {total} filas ---")
    print(f"  grosero=1:     {df['grosero'].sum():>4} ({df['grosero'].mean()*100:.1f}%)")
    print(f"  amenaza=1:     {df['amenaza'].sum():>4} ({df['amenaza'].mean()*100:.1f}%)")
    print(f"  inapropiado=1: {df['inapropiado'].sum():>4} ({df['inapropiado'].mean()*100:.1f}%)")


def main():
    ruta_entrada = os.path.join(INPUT_DIR, "dataset_final_entrenamiento.csv")
    if not os.path.exists(ruta_entrada):
        print(f"No se encontro {ruta_entrada}. Corre primero la celda de combinar dataset final.")
        return

    df = pd.read_csv(ruta_entrada)
    print(f"Dataset de entrada: {len(df)} filas")

    df["estrato"] = df.apply(construir_estrato, axis=1)
    print("\nDistribucion de estratos (categoria mas rara por fila):")
    print(df["estrato"].value_counts())

    # Verificamos que cada estrato tenga al menos unas pocas filas, o el
    # split estratificado va a fallar. Si algun estrato tiene muy pocas
    # filas, avisamos (aunque con 33 de inapropiado deberia alcanzar).
    conteo_estratos = df["estrato"].value_counts()
    estrato_minimo = conteo_estratos.min()
    if estrato_minimo < 10:
        print(f"\n[AVISO] El estrato mas pequeno tiene solo {estrato_minimo} filas. "
              f"El split igual funcionara, pero revisa que val/test no queden con 0-1 ejemplos.")

    # --- Split 1: separamos TRAIN del resto (val+test) ---
    prop_resto = PROP_VAL + PROP_TEST
    df_train, df_resto = train_test_split(
        df,
        test_size=prop_resto,
        stratify=df["estrato"],
        random_state=RANDOM_STATE,
    )

    # --- Split 2: dividimos el resto en VALIDATION y TEST ---
    prop_test_relativa = PROP_TEST / prop_resto  # proporcion de test DENTRO del resto
    df_val, df_test = train_test_split(
        df_resto,
        test_size=prop_test_relativa,
        stratify=df_resto["estrato"],
        random_state=RANDOM_STATE,
    )

    # Quitamos la columna auxiliar antes de guardar
    for d in (df_train, df_val, df_test):
        d.drop(columns=["estrato"], inplace=True)

    # --- Reporte de balance en cada split ---
    reportar_balance("TRAIN", df_train)
    reportar_balance("VALIDATION", df_val)
    reportar_balance("TEST", df_test)

    # --- Guardar ---
    ruta_train = os.path.join(OUTPUT_DIR, "train.csv")
    ruta_val = os.path.join(OUTPUT_DIR, "validation.csv")
    ruta_test = os.path.join(OUTPUT_DIR, "test.csv")

    df_train.to_csv(ruta_train, index=False, encoding="utf-8")
    df_val.to_csv(ruta_val, index=False, encoding="utf-8")
    df_test.to_csv(ruta_test, index=False, encoding="utf-8")

    print(f"\n-> Guardado: {ruta_train} ({len(df_train)} filas)")
    print(f"-> Guardado: {ruta_val} ({len(df_val)} filas)")
    print(f"-> Guardado: {ruta_test} ({len(df_test)} filas)")

    print("""
SIGUIENTE PASO:
Estos tres archivos (train.csv, validation.csv, test.csv) estan listos
para cargarse con la libreria `datasets` de HuggingFace y tokenizarse
para el fine-tuning. Revisa arriba que 'amenaza' e 'inapropiado' tengan
al menos unos pocos ejemplos en validation y test -- si alguno quedo en
0, considera bajar PROP_VAL/PROP_TEST o revisar el balance del dataset
combinado antes de continuar.
""")


if __name__ == "__main__":
    main()

## **Cargar datasets** (Entrenamiento, validación y test)

In [ ]:
from datasets import load_dataset

DATASET_DIR = str(PROCESSED_DIR)

data_files = {
    "train": f"{DATASET_DIR}/train.csv",
    "validation": f"{DATASET_DIR}/validation.csv",
    "test": f"{DATASET_DIR}/test.csv",
}
dataset = load_dataset("csv", data_files=data_files)

# Construimos la columna 'labels' como vector [grosero, amenaza, inapropiado]
# porque asi es como el modelo de HuggingFace espera las etiquetas multilabel.
LABEL_COLUMNS = ["grosero", "amenaza", "inapropiado"]

def construir_labels(ejemplo):
    ejemplo["labels"] = [float(ejemplo[col]) for col in LABEL_COLUMNS]
    return ejemplo

dataset = dataset.map(construir_labels)
print(dataset)
print(dataset["train"][0])

## **Tokenización**

In [ ]:
# IMPORTANT: If you encounter 'RuntimeError: generic_type: cannot initialize type "RpcBackendOptions"',
# it means torch was reinstalled in a live session. Please RESTART THE KERNEL (Kernel -> Restart Kernel)
# and then run this cell again from scratch.

import os
import torch


from transformers import AutoTokenizer

# Partimos del modelo ya adaptado a hate speech en espanol (Ruta A que
# definimos), en vez de BETO puro -- ya tiene nocion previa de toxicidad.
MODEL_NAME = "piuba-bigdata/beto-contextualized-hate-speech"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenizar(ejemplo):
    return tokenizer(ejemplo["texto"], truncation=True, padding="max_length", max_length=128)

dataset_tokenizado = dataset.map(tokenizar, batched=True)

# Limpiamos columnas que el modelo no necesita, dejamos solo lo indispensable
columnas_a_quitar = ["texto", "fuente"] + LABEL_COLUMNS
columnas_a_quitar = [c for c in columnas_a_quitar if c in dataset_tokenizado["train"].column_names]
dataset_tokenizado = dataset_tokenizado.remove_columns(columnas_a_quitar)
dataset_tokenizado.set_format("torch")

## **Cargar modelo configurado para multilabel**

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLUMNS),
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True,  # el head original no coincide con tus 3 etiquetas, se reinicializa
)

## **Pesos por clase** (compensación de pesos)

In [ ]:
import numpy as np
from torch.nn import BCEWithLogitsLoss
from transformers import Trainer

train_labels = np.array(dataset["train"]["labels"])
n_total = len(train_labels)
n_positivos = train_labels.sum(axis=0)
pos_weight_crudo = (n_total - n_positivos) / n_positivos

TOPES_POR_CATEGORIA = {                                     # <- aqui defines el limite maximo
    "grosero": 3.0,
    "amenaza": 8.0,        # se queda igual, funciono bien
    "inapropiado": 3.5,
}

# Capamos el peso maximo para que no se vuelva excesivamente agresivo
topes_array = np.array([TOPES_POR_CATEGORIA[col] for col in LABEL_COLUMNS])
pos_weight = np.minimum(pos_weight_crudo, topes_array)   # <- aqui se aplica

pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float)
print("pos_weight crudo:", pos_weight_crudo)
print("pos_weight capado (usado):", pos_weight_tensor)

class TrainerConPesos(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## **Métricas de evaluación**

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits))
    preds = (probs > 0.5).int().numpy()

    resultado = {
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
    }
    for i, nombre in enumerate(LABEL_COLUMNS):
        resultado[f"f1_{nombre}"] = f1_score(labels[:, i], preds[:, i], zero_division=0)
        resultado[f"precision_{nombre}"] = precision_score(labels[:, i], preds[:, i], zero_division=0)
        resultado[f"recall_{nombre}"] = recall_score(labels[:, i], preds[:, i], zero_division=0)
    return resultado

## **Configuración del entrenamiento**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_ARTIFACTS_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_inapropiado",
    logging_steps=20,
    report_to="none",
)

## **Entrenamiento**

In [ ]:
trainer = TrainerConPesos(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado["train"],
    eval_dataset=dataset_tokenizado["validation"],
    compute_metrics=compute_metrics,
)

trainer.train()

## **Test**

In [ ]:
resultados_test = trainer.evaluate(dataset_tokenizado["test"])
for k, v in resultados_test.items():
    print(f"{k}: {v:.4f}")

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

predicciones = trainer.predict(dataset_tokenizado["test"])
probs = torch.sigmoid(torch.tensor(predicciones.predictions))
preds = (probs > 0.5).int().numpy()
labels_reales = predicciones.label_ids

matrices = multilabel_confusion_matrix(labels_reales, preds)
for i, nombre in enumerate(LABEL_COLUMNS):
    print(f"\n--- {nombre} ---")
    print("        Pred 0   Pred 1")
    print(f"Real 0  {matrices[i][0][0]:>6}   {matrices[i][0][1]:>6}")
    print(f"Real 1  {matrices[i][1][0]:>6}   {matrices[i][1][1]:>6}")

## Guardar

In [ ]:
RUTA_MODELO_FINAL = str(MODEL_ARTIFACTS_DIR / "modelo_moderacion_final")
trainer.save_model(RUTA_MODELO_FINAL)
tokenizer.save_pretrained(RUTA_MODELO_FINAL)
print(f"Modelo guardado en: {RUTA_MODELO_FINAL}")

## **Probar textos**

In [ ]:
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Celda pensada para poder correrse de forma independiente (ej. tras
# reiniciar el kernel), por eso vuelve a resolver las rutas del proyecto.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
MODEL_ARTIFACTS_DIR = PROJECT_ROOT / "model_artifacts"

RUTA_MODELO_FINAL = str(MODEL_ARTIFACTS_DIR / "modelo_moderacion_final")

model = AutoModelForSequenceClassification.from_pretrained(RUTA_MODELO_FINAL)
tokenizer = AutoTokenizer.from_pretrained(RUTA_MODELO_FINAL)

# Mover al mismo device donde vayas a hacer inferencia
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print(f"Modelo cargado desde {RUTA_MODELO_FINAL}, corriendo en: {device}")

In [ ]:
UMBRALES_POR_CATEGORIA = {
    "grosero": 0.5,
    "amenaza": 0.5,
    "inapropiado": 0.30,  # mas bajo, priorizando recall (tolerancia cero)
}

def moderar_texto(texto):
    inputs = tokenizer(texto, return_tensors="pt", truncation=True,
                        padding="max_length", max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    resultado = {}
    for i, nombre in enumerate(LABEL_COLUMNS):
        umbral = UMBRALES_POR_CATEGORIA[nombre]
        resultado[nombre] = {
            "probabilidad": round(float(probs[i]), 4),
            "activado": bool(probs[i] > umbral),
        }

    bloqueado = any(resultado[cat]["activado"] for cat in LABEL_COLUMNS)

    print(f"\nTexto: \"{texto}\"")
    print("-" * 60)
    for nombre in LABEL_COLUMNS:
        marca = "🔴" if resultado[nombre]["activado"] else "🟢"
        print(f"  {marca} {nombre:15s} probabilidad: {resultado[nombre]['probabilidad']:.4f} (umbral: {UMBRALES_POR_CATEGORIA[nombre]})")
    print("-" * 60)
    print(f"  DECISION FINAL: {'❌ BLOQUEAR' if bloqueado else '✅ PERMITIR'}")

    return resultado

In [ ]:
ejemplos_de_prueba = [
    "Hola caras de mi pene, les ofresco unas latas de caguama",
    "Buen servicio, me atendieron muy bien, completamente recomendado",
    "Ofrezco unas latas de aluminio, los pueden recoger en mi casa",
    "Pinche lugar de cagada, lo compran muy barato los malditos",
    "Pinches putos no me pagaron, ojala les asalten el establecimiento",
    "Tengo casa sola, vengan traviesos 😘",
    "Pesimo servicio, el personal no me atendio",
    "Pvt0s malditos este lugar no funciona nunca",  # variante ofuscada
]

for texto in ejemplos_de_prueba:
    moderar_texto(texto)